# Import and Setup

In [7]:
import os
import json
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"
REASONING_EFFORT = "medium"  # "low", "medium", "high", or None to disable

# Generation Functions

- Given a notebook, in the last cell of the notebook replace the description = "INSERT TEXT HERE ABOUT NOTEBOOK process" line with a full description of the notebook.

- In the last cell of the notebook replace properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} with a dictionary of properties and category that the notebook falls under.

- For each other instance of description = "INSERT TEXT HERE ABOUT dataset_name", replace the text with a full description of the dataset.

- For each other instance of properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} replace the text with properties of the dataset.

Assume that a future data scientist would like to access the descriptions and properties and think about what would be useful for them to know.

Assume that they might access each description and property separately. 

Each bullet point should be a separate call to the OpenAI model. 

Generate a function, given a notebook url, apply each bulllet point using an openai model (with reasoning parameter) and save new copy of notebook


In [12]:
import json
import re
import os
import shutil


def _call_with_reasoning(messages: list) -> str:
    """Call the OpenAI model with optional reasoning."""
    extra = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT is not None else {}
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **extra,
    )
    return response.choices[0].message.content


# Compiled once; shared by all calls to fill_notebook_descriptions.
#
# Matches the notebook-level description line, which always contains "process"
# after the notebook name. Observed variants across batch_4_clean:
#   "INSERT TEXT HERE ABOUT mrpc_paraphrase_inference_distilbert process"
#   "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_margin_rule process/notebook"
#   "INSERT TEXT HERE ABOUT flan_t5_answer_prefix_yes_no_mrpc.ipynb process/notebook"
# [^"]+ matches the name (including dots), [^"]* absorbs anything after "process".
_PROCESS_DESC_PAT = re.compile(
    r'description\s*=\s*"INSERT TEXT HERE ABOUT [^"]+ process[^"]*"'
)
# Matches dataset-level description lines.
# [\w-]+ handles both underscore names (glue_mrpc_validation) and
# hyphenated names (distilbert-margin-and-prediction, sentence-transformers-sentence-1).
# The name always ends immediately before a closing quote, so it never
# accidentally matches the process-cell forms (which have a space before "process").
_DATASET_DESC_PAT = re.compile(
    r'description\s*=\s*"INSERT TEXT HERE ABOUT ([\w-]+)"'
)
# Matches: properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"}
_PROPERTIES_PAT = re.compile(
    r'properties\s*=\s*\{"INSERT_PROPERTY":\s*"INSERT_CATEGORIES"\}'
)


def _fill_dataset_placeholders(src: str, notebook_content: str) -> str:
    """Fill all dataset description and properties placeholders in a source string."""
    desc_matches = list(_DATASET_DESC_PAT.finditer(src))
    prop_matches = list(_PROPERTIES_PAT.finditer(src))

    # One LLM call per description placeholder.
    desc_replacements = []
    for m in desc_matches:
        dataset_name = m.group(1)
        print(f"Bullet 3: generating description for '{dataset_name}' …")
        ds_desc = _call_with_reasoning([{
            "role": "user",
            "content": (
                "You are a data scientist documenting a dataset used in a machine "
                "learning notebook.\n\n"
                f"Notebook context:\n{notebook_content}\n\n"
                f"Write a concise description of the dataset named '{dataset_name}' "
                "as used in this notebook. Include what data it contains, its structure "
                "(columns/fields), and its role in this workflow. A future data scientist "
                "will access this description separately to understand this dataset.\n\n"
                "Return ONLY the description text, no extra formatting or quotes."
            ),
        }]).strip()
        desc_replacements.append(f"description = {json.dumps(ds_desc)}")

    if desc_replacements:
        desc_iter = iter(desc_replacements)
        src = _DATASET_DESC_PAT.sub(lambda _: next(desc_iter), src)

    # One LLM call per properties placeholder, paired with desc by index.
    props_replacements = []
    for j, _ in enumerate(prop_matches):
        ctx_name = desc_matches[j].group(1) if j < len(desc_matches) else "the dataset in this cell"
        print(f"Bullet 4: generating properties for '{ctx_name}' …")
        ds_props_raw = _call_with_reasoning([{
            "role": "user",
            "content": (
                "You are a data scientist documenting a dataset used in a machine "
                "learning notebook.\n\n"
                f"Notebook context:\n{notebook_content}\n\n"
                f"Generate a Python dictionary of properties for the dataset named "
                f"'{ctx_name}'. Properties should help a future data scientist filter "
                "or find this dataset by key attributes.\n"
                'Example: {"task": "paraphrase detection", "domain": "news", '
                '"split": "validation", "size": "408", "source": "glue/mrpc"}\n\n'
                "Return ONLY a single-line valid Python dictionary literal, no explanation."
            ),
        }]).strip()
        props_replacements.append(f"properties = {ds_props_raw}")

    if props_replacements:
        props_iter = iter(props_replacements)
        src = _PROPERTIES_PAT.sub(lambda _: next(props_iter), src)

    return src


def fill_notebook_descriptions(notebook_path: str, output_path: str = None) -> str:
    """
    Fill in description and properties placeholders in a notebook using four
    separate LLM calls (one per bullet point per occurrence), then save the result.

    Bullet 1 — notebook-level description  (last cell's description placeholder)
    Bullet 2 — notebook-level properties   (last cell's properties placeholder)
    Bullet 3 — per-dataset description     (one call per description placeholder)
    Bullet 4 — per-dataset properties      (one call per properties placeholder,
                                            paired with its corresponding dataset)

    Most cells have one description+properties pair. A few embedding cells have
    two pairs (e.g. sentence-1 and sentence-2 embeddings), and each gets its own
    LLM call.

    If the last (process) cell also contains dataset placeholders *before* the
    process description line, those are handled as Bullets 3 & 4 first, then the
    process description and properties are filled (Bullets 1 & 2).
    """
    if output_path is None:
        output_path = notebook_path

    with open(notebook_path, "r") as f:
        nb = json.load(f)

    cells = nb.get("cells", [])

    # Join source lists into plain strings for each cell.
    sources = ["".join(cell.get("source", [])) for cell in cells]

    # Locate the last cell that holds the notebook-level ("process") description.
    last_process_idx = None
    for i, src in enumerate(sources):
        if _PROCESS_DESC_PAT.search(src):
            last_process_idx = i

    # Identify dataset cells (all cells with a placeholder that are NOT the process cell).
    dataset_idxs = [
        i for i, src in enumerate(sources)
        if i != last_process_idx
        and (_DATASET_DESC_PAT.search(src) or _PROPERTIES_PAT.search(src))
    ]

    if last_process_idx is None and not dataset_idxs:
        print("No placeholders found — skipping.")
        return output_path

    # Build full notebook text for LLM context (source only, no outputs).
    notebook_content = "\n\n---\n\n".join(src for src in sources if src.strip())

    # ── Process cell (last_process_idx) ───────────────────────────────────────
    if last_process_idx is not None:
        src = sources[last_process_idx]
        proc_match = _PROCESS_DESC_PAT.search(src)
        proc_start = proc_match.start()

        # Split: everything before the process description may contain dataset placeholders.
        pre_src = src[:proc_start]
        proc_src = src[proc_start:]

        # Bullets 3 & 4 for any dataset placeholders living before the process line.
        if _DATASET_DESC_PAT.search(pre_src) or _PROPERTIES_PAT.search(pre_src):
            pre_src = _fill_dataset_placeholders(pre_src, notebook_content)

        # Bullet 1: notebook-level description.
        print("Bullet 1: generating notebook description …")
        nb_desc = _call_with_reasoning([{
            "role": "user",
            "content": (
                "You are a data scientist documenting a Jupyter notebook workflow.\n\n"
                f"Notebook content:\n{notebook_content}\n\n"
                "Write a concise description of what this notebook does — its purpose, "
                "the ML task, the model used, the dataset, and the overall process. "
                "A future data scientist will access this description separately to "
                "understand the notebook workflow.\n\n"
                "Return ONLY the description text, no extra formatting or quotes."
            ),
        }]).strip()

        # Bullet 2: notebook-level properties.
        print("Bullet 2: generating notebook properties …")
        nb_props_raw = _call_with_reasoning([{
            "role": "user",
            "content": (
                "You are a data scientist documenting a Jupyter notebook workflow.\n\n"
                f"Notebook content:\n{notebook_content}\n\n"
                "Generate a Python dictionary of properties that categorize this notebook. "
                "Properties should help a future data scientist filter or find this notebook "
                "by key attributes.\n"
                'Example: {"task": "paraphrase detection", "model": "distilbert-base-uncased-MRPC", '
                '"dataset": "glue/mrpc", "evaluation": "accuracy, f1-score"}\n\n'
                "Return ONLY a single-line valid Python dictionary literal, no explanation."
            ),
        }]).strip()

        proc_src = _PROCESS_DESC_PAT.sub(
            lambda _: f"description = {json.dumps(nb_desc)}", proc_src
        )
        proc_src = _PROPERTIES_PAT.sub(
            lambda _: f"properties = {nb_props_raw}", proc_src
        )

        cells[last_process_idx]["source"] = pre_src + proc_src

    # ── Bullets 3 & 4: pure dataset cells ─────────────────────────────────────
    for i in dataset_idxs:
        src = sources[i]
        src = _fill_dataset_placeholders(src, notebook_content)
        cells[i]["source"] = src

    nb["cells"] = cells
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(nb, f, indent=1)

    print(f"Saved → {output_path}")
    return output_path

# Notebook Transformation

- First cell copy all the notebooks in batch_4_clean to batch_4_description
- Second cell apply the generation function to the batch_4_description/mrpc_paraphrase_inference_distilbert.ipynb notebook to test.
- Third cell apply the function to all notebooks in batch_4_description.

In [13]:
import glob as glob_module

src_dir = "../notebooks/batch_4_clean"
dst_dir = "../notebooks/batch_4_description"

if os.path.exists(dst_dir):
    shutil.rmtree(dst_dir)
shutil.copytree(src_dir, dst_dir)

all_notebooks = [
    p for p in glob_module.glob(f"{dst_dir}/**/*.ipynb", recursive=True)
    if ".ipynb_checkpoints" not in p
]
print(f"Copied {src_dir} → {dst_dir}")
print(f"Found {len(all_notebooks)} notebooks")

Copied ../notebooks/batch_4_clean → ../notebooks/batch_4_description
Found 43 notebooks


In [14]:
# Test on a single notebook first
test_path = f"{dst_dir}/mrpc_paraphrase_inference_distilbert.ipynb"
fill_notebook_descriptions(test_path)

Bullet 1: generating notebook description …
Bullet 2: generating notebook properties …
Bullet 3: generating description for 'glue_mrpc_validation' …
Bullet 4: generating properties for 'glue_mrpc_validation' …
Bullet 3: generating description for 'mrpc_distilbert_predictions' …
Bullet 4: generating properties for 'mrpc_distilbert_predictions' …
Bullet 3: generating description for 'mrpc_paraphrase_inference_distilbert_mistakes' …
Bullet 4: generating properties for 'mrpc_paraphrase_inference_distilbert_mistakes' …
Bullet 3: generating description for 'mrpc_paraphrase_inference_distilbert_output' …
Bullet 4: generating properties for 'mrpc_paraphrase_inference_distilbert_output' …
Saved → ../notebooks/batch_4_description/mrpc_paraphrase_inference_distilbert.ipynb


'../notebooks/batch_4_description/mrpc_paraphrase_inference_distilbert.ipynb'

In [15]:
# Apply to all notebooks in batch_4_description
errors = []
for i, nb_path in enumerate(all_notebooks):
    print(f"\n[{i+1}/{len(all_notebooks)}] {nb_path}")
    try:
        fill_notebook_descriptions(nb_path)
    except Exception as e:
        print(f"  ERROR: {e}")
        errors.append((nb_path, str(e)))

print(f"\nDone. {len(all_notebooks) - len(errors)} succeeded, {len(errors)} failed.")
for path, err in errors:
    print(f"  FAILED: {path}\n    {err}")


[1/43] ../notebooks/batch_4_description/mrpc_paraphrase_inference_distilbert.ipynb
No placeholders found — skipping.

[2/43] ../notebooks/batch_4_description/variations_mrpc_paraphrase_inference_distilbert/zero_shot_nli_pipeline_mrpc.ipynb
Bullet 1: generating notebook description …
Bullet 2: generating notebook properties …
Bullet 3: generating description for 'mrpc_zero_shot_predictions' …
Bullet 4: generating properties for 'mrpc_zero_shot_predictions' …
Bullet 3: generating description for 'zero_shot_nli_pipeline_mrpc_mistakes' …
Bullet 4: generating properties for 'zero_shot_nli_pipeline_mrpc_mistakes' …
Bullet 3: generating description for 'zero_shot_nli_pipeline_mrpc_output' …
Bullet 4: generating properties for 'zero_shot_nli_pipeline_mrpc_output' …
Saved → ../notebooks/batch_4_description/variations_mrpc_paraphrase_inference_distilbert/zero_shot_nli_pipeline_mrpc.ipynb

[3/43] ../notebooks/batch_4_description/variations_mrpc_paraphrase_inference_distilbert/hf_pipeline_sequenc